<a href="https://colab.research.google.com/github/satish24011993/mlops_project/blob/master/SQL_Training_Notebook_BrowseJobs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗄️ SQL for Data Engineering — Instructor Training Notebook
**BrowseJobs Technologies · Complete Course Companion**

This notebook is organised into **14 chapters**. Every query is **fully runnable** — it uses SQLite through the `%%sql` magic, so students can execute and experiment live instead of just reading SQL.

Each chapter has:
- **Concept** — a short explanation you can teach from
- **Runnable queries** — press `Shift + Enter` and see real results
- **🧪 Class Practice** — exercises for students
- **💬 Interview Corner** — the questions that come up in L1/L2 rounds

> ⚠️ **Run the SETUP cells first** (Chapter 3) — they create the database every chapter uses.


## 📚 Table of Contents
1. Databases — SQL vs NoSQL, CRUD, the Relational Model
2. Keys & Constraints
3. Creating Tables & Inserting Data (⚙️ SETUP — run first!)
4. SELECT Fundamentals — WHERE · ORDER BY · LIMIT · Alias
5. Filtering Operators — LIKE & Wildcards · IN · BETWEEN
6. Aggregations — COUNT · SUM · AVG · GROUP BY · HAVING
7. Joins — Inner · Left · Right · Full · Self · Cross
8. The C-T-A-G Query-Writing Framework
9. Subqueries & CTEs
10. Window Functions — RANK · DENSE_RANK · ROW_NUMBER · LAG · LEAD
11. CASE Statements & Stored Procedures
12. DELETE vs TRUNCATE vs DROP · UNION vs UNION ALL · Handling Duplicates
13. Query Optimisation — Indexing · Views · Partitioning · Execution Order
14. Data Modelling — OLTP vs OLAP · Star vs Snowflake · Normalisation


---
# Chapter 1 · Databases — SQL vs NoSQL

### SQL (Structured Query Language) databases
- Data is stored in **tables** with rows and columns.
- **Relational model** — many tables related to each other via keys.
- **Fixed schema** — the structure of the table is predefined and strictly followed.
- **Vertical scalability** — scale by adding more RAM/CPU/storage to the server.
- Examples: **MySQL, PostgreSQL, SQL Server, Oracle**.
- Use case: **transactional systems** — banking, social media, e-commerce.

### NoSQL databases
- Data stored in other formats: **key:value pairs (JSON)**, documents, graphs.
- **Dynamic / flexible schema** — schema can evolve.
- **Horizontal scalability** — scale using distributed architecture (add more machines).
- Example: **MongoDB** (BSON query language).
- Use cases: **big data, analytics, AI/ML**.

### CRUD — the 4 operations every database supports
**C**reate · **R**ead · **U**pdate · **D**elete

### Fact vs Dimension tables (data-warehouse vocabulary)
| Table type | Contains | Examples |
|---|---|---|
| **Fact** | Quantitative, measurable data | sales, revenue, quantity |
| **Factless fact** | Events with no measures | attendance: date, studentid, courseid |
| **Dimension** | Descriptive data that defines facts | customer, product, logistics |


---
# Chapter 2 · Keys & Constraints

| Key / Constraint | Unique? | NULLs allowed? | How many per table? |
|---|---|---|---|
| **Primary key** | ✅ Must be unique | ❌ Never | 1 (can be composite) |
| **Foreign key** | ❌ Duplicates OK | ✅ Yes | Any number |
| **Unique key** | ✅ Must be unique | ✅ Yes* | Any number |
| **Not null** | — | ❌ Never | Per column |
| **Composite key** | ✅ Combination unique | ❌ | e.g. (orderid + productid) |

*NULL handling in UNIQUE differs by engine: **SQL Server** treats `null = null` → only ONE null allowed. **PostgreSQL / MySQL** treat `null <> null` → multiple nulls allowed. (Great interview trivia!)

**More key vocabulary:**
- **Surrogate key** — system-generated value used as primary key (auto-increment id).
- **Natural key** — given by the business (aadhar number, email).
- **Candidate key** — any column (or set) that could uniquely identify a row; the primary key is chosen from the candidates.

**Example:** a `sales` table where `customerid` and `productid` are foreign keys pointing to the `customer` and `product` dimension tables.


---
# Chapter 3 · ⚙️ SETUP — Create the Training Database

Run the next cells **in order, once**. They install the SQL magic, create a SQLite database, and load the HR schema used throughout the course: **Employees · Departments · Jobs · Job_History**.


In [1]:
# ⚙️ SETUP 1 — install & load the SQL magic (one time)
!pip install -q ipython-sql prettytable
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%load_ext sql
%config SqlMagic.autopandas = False
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# create / connect to the training database
%sql sqlite:///browsejobs_sql.db
print('✅ Connected to browsejobs_sql.db')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.1 MB/s eta 0:00:00
✅ Connected to browsejobs_sql.db


In [2]:
%%sql
-- ⚙️ SETUP 2 — create the four tables (drop first so the cell is re-runnable)
DROP TABLE IF EXISTS Job_History;
DROP TABLE IF EXISTS Employees;
DROP TABLE IF EXISTS Departments;
DROP TABLE IF EXISTS Jobs;

CREATE TABLE Departments (
    department_id INT PRIMARY KEY,
    department_name VARCHAR(100),
    location VARCHAR(100),
    manager_id INT
);

CREATE TABLE Jobs (
    job_id VARCHAR(10) PRIMARY KEY,
    job_title VARCHAR(255),
    min_salary DECIMAL(10, 2),
    max_salary DECIMAL(10, 2)
);

CREATE TABLE Employees (
    employee_id INT PRIMARY KEY,
    first_name VARCHAR(255),
    last_name VARCHAR(255),
    email VARCHAR(255),
    phone_number VARCHAR(20),
    hire_date DATE,
    job_id VARCHAR(10),
    salary DECIMAL(10, 2),
    department_id INT,
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY (job_id) REFERENCES Jobs(job_id)
);

CREATE TABLE Job_History (
    employee_id INT,
    start_date DATE,
    end_date DATE,
    job_id VARCHAR(10),
    department_id INT,
    PRIMARY KEY (employee_id, start_date),          -- composite primary key!
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id),
    FOREIGN KEY (job_id) REFERENCES Jobs(job_id),
    FOREIGN KEY (department_id) REFERENCES Departments(department_id)
);

[]

In [3]:
%%sql
-- ⚙️ SETUP 3 — insert the data
INSERT INTO Departments (department_id, department_name, location, manager_id) VALUES
(10, 'Development', 'New York', 101),
(20, 'Human Resources', 'Chicago', 102),
(30, 'Finance', 'Boston', 103),
(40, 'Marketing', 'Los Angeles', 104),
(50, 'IT', 'San Francisco', 109),
(60, 'Support', 'Remote', NULL);          -- no employees yet

INSERT INTO Jobs (job_id, job_title, min_salary, max_salary) VALUES
('DEV01', 'Senior Developer', 70000.00, 100000.00),
('DEV02', 'Lead Developer', 90000.00, 130000.00),
('HR01', 'HR Manager', 60000.00, 90000.00),
('HR02', 'HR Specialist', 50000.00, 75000.00),
('FIN01', 'Financial Analyst', 75000.00, 110000.00),
('FIN02', 'Senior Accountant', 80000.00, 120000.00),
('MKT01', 'Marketing Specialist', 55000.00, 80000.00),
('MKT02', 'Marketing Manager', 65000.00, 95000.00),
('IT01', 'IT Manager', 85000.00, 125000.00),
('IT02', 'Network Engineer', 78000.00, 115000.00);

INSERT INTO Employees VALUES
(101, 'John', 'Doe', 'john.doe@example.com', '123-456-7890', '2020-01-15', 'DEV01', 75000.00, 10),
(102, 'Jane', 'Smith', 'jane.smith@example.com', '234-567-8901', '2019-03-22', 'HR01', 68000.00, 20),
(103, 'Robert', 'Brown', 'robert.brown@example.com', '345-678-9012', '2018-07-01', 'FIN01', 85000.00, 30),
(104, 'Emily', 'Davis', 'emily.davis@example.com', '456-789-0123', '2021-11-10', 'MKT01', 62000.00, 40),
(105, 'Michael', 'Wilson', 'michael.wilson@example.com', '567-890-1234', '2022-06-18', 'DEV02', 78000.00, 10),
(106, 'Olivia', 'Taylor', 'olivia.taylor@example.com', '678-901-2345', '2020-04-12', 'HR02', 71000.00, 20),
(107, 'William', 'Moore', 'william.moore@example.com', '789-012-3456', '2017-09-25', 'FIN02', 90000.00, 30),
(108, 'Sophia', 'Anderson', 'sophia.anderson@example.com', '890-123-4567', '2023-02-14', 'MKT02', 65000.00, 40),
(109, 'James', 'Thomas', 'james.thomas@example.com', '901-234-5678', '2016-12-05', 'IT01', 87000.00, 50),
(110, 'Isabella', 'Jackson', 'isabella.jackson@example.com', '012-345-6789', '2019-08-30', 'IT02', 82000.00, 50);

INSERT INTO Job_History VALUES
(101, '2020-01-15', '2021-12-31', 'DEV01', 10),
(101, '2022-01-01', NULL, 'DEV02', 20),     -- current job
(102, '2019-03-22', NULL, 'HR01', 20),
(103, '2018-07-01', '2023-06-30', 'FIN01', 30),
(103, '2023-07-01', NULL, 'FIN02', 40),
(104, '2021-11-10', NULL, 'MKT01', 40),
(105, '2022-06-18', NULL, 'DEV02', 10),
(106, '2020-04-12', NULL, 'HR02', 20),
(107, '2017-09-25', NULL, 'FIN02', 30),
(108, '2023-02-14', NULL, 'MKT02', 40),
(109, '2016-12-05', NULL, 'IT01', 50),
(110, '2019-08-30', NULL, 'IT02', 50);

[]

In [4]:
%%sql
-- ✅ Sanity check — you should see 10 employees
SELECT COUNT(*) AS employee_count FROM Employees;

employee_count
10


---
# Chapter 4 · SELECT Fundamentals

The **read** in CRUD. `SELECT` extracts data; `WHERE` filters rows; `ORDER BY` sorts; `LIMIT` caps the row count.


In [5]:
%%sql
-- Select ALL data from the employees table
SELECT * FROM Employees;

employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,department_id
101,John,Doe,john.doe@example.com,123-456-7890,2020-01-15,DEV01,75000,10
102,Jane,Smith,jane.smith@example.com,234-567-8901,2019-03-22,HR01,68000,20
103,Robert,Brown,robert.brown@example.com,345-678-9012,2018-07-01,FIN01,85000,30
104,Emily,Davis,emily.davis@example.com,456-789-0123,2021-11-10,MKT01,62000,40
105,Michael,Wilson,michael.wilson@example.com,567-890-1234,2022-06-18,DEV02,78000,10
106,Olivia,Taylor,olivia.taylor@example.com,678-901-2345,2020-04-12,HR02,71000,20
107,William,Moore,william.moore@example.com,789-012-3456,2017-09-25,FIN02,90000,30
108,Sophia,Anderson,sophia.anderson@example.com,890-123-4567,2023-02-14,MKT02,65000,40
109,James,Thomas,james.thomas@example.com,901-234-5678,2016-12-05,IT01,87000,50
110,Isabella,Jackson,isabella.jackson@example.com,012-345-6789,2019-08-30,IT02,82000,50


In [ ]:
%%sql
-- Select SPECIFIC columns (best practice — avoid SELECT * in production!)
SELECT first_name, last_name, salary FROM Employees;

In [7]:
%%sql
-- WHERE: filter rows on a condition
SELECT * FROM Employees
WHERE salary > 75000;

employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,department_id
103,Robert,Brown,robert.brown@example.com,345-678-9012,2018-07-01,FIN01,85000,30
105,Michael,Wilson,michael.wilson@example.com,567-890-1234,2022-06-18,DEV02,78000,10
107,William,Moore,william.moore@example.com,789-012-3456,2017-09-25,FIN02,90000,30
109,James,Thomas,james.thomas@example.com,901-234-5678,2016-12-05,IT01,87000,50
110,Isabella,Jackson,isabella.jackson@example.com,012-345-6789,2019-08-30,IT02,82000,50


In [6]:
%%sql
-- AND: both conditions must be true
SELECT * FROM Employees
WHERE hire_date > '2020-01-01' AND salary > 75000;

employee_id,first_name,last_name,email,phone_number,hire_date,job_id,salary,department_id
105,Michael,Wilson,michael.wilson@example.com,567-890-1234,2022-06-18,DEV02,78000,10


In [8]:
%%sql
-- OR: at least one condition true
SELECT first_name, last_name, salary FROM Employees
WHERE salary > 75000 OR first_name = 'Michael';

first_name,last_name,salary
Robert,Brown,85000
Michael,Wilson,78000
William,Moore,90000
James,Thomas,87000
Isabella,Jackson,82000


In [9]:
%%sql
-- NOT: the opposite of the where condition
SELECT first_name, last_name, salary FROM Employees
WHERE NOT salary > 75000;

first_name,last_name,salary
John,Doe,75000
Jane,Smith,68000
Emily,Davis,62000
Olivia,Taylor,71000
Sophia,Anderson,65000


In [10]:
%%sql
-- ORDER BY: ascending is the DEFAULT
SELECT first_name, last_name, salary FROM Employees
ORDER BY salary;

first_name,last_name,salary
Emily,Davis,62000
Sophia,Anderson,65000
Jane,Smith,68000
Olivia,Taylor,71000
John,Doe,75000
Michael,Wilson,78000
Isabella,Jackson,82000
Robert,Brown,85000
James,Thomas,87000
William,Moore,90000


In [11]:
%%sql
-- Descending order + LIMIT: top earner
SELECT first_name, last_name, salary FROM Employees
ORDER BY salary DESC
LIMIT 1;

first_name,last_name,salary
William,Moore,90000


In [12]:
%%sql
-- Alias with AS: rename columns in the output
SELECT first_name AS fname, last_name AS lname, salary AS sal
FROM Employees;

fname,lname,sal
John,Doe,75000
Jane,Smith,68000
Robert,Brown,85000
Emily,Davis,62000
Michael,Wilson,78000
Olivia,Taylor,71000
William,Moore,90000
Sophia,Anderson,65000
James,Thomas,87000
Isabella,Jackson,82000


### 🧪 Class Practice 4
1. All employees hired before 2019.
2. First name + salary of employees earning between 70k and 85k (use `AND` for now).
3. The 3 lowest-paid employees.
4. Everyone NOT in department 10.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 5 · Filtering Operators — LIKE, IN, BETWEEN

### Wildcards for LIKE
| Wildcard | Meaning |
|---|---|
| `%` | 0 or more characters |
| `_` | exactly ONE character |
| `[aeiou]` | one character from the set (SQL Server) |
| `SIMILAR TO '(J\|K)%'` | regex-style (PostgreSQL) |


In [ ]:
%%sql
-- Names starting with M   →  'M%'
SELECT * FROM Employees WHERE first_name LIKE 'M%';

In [ ]:
%%sql
-- Names ENDING with n     →  '%n'
SELECT * FROM Employees WHERE first_name LIKE '%n';

In [ ]:
%%sql
-- Exactly 4 characters    →  '____'  (four underscores)
SELECT first_name FROM Employees WHERE first_name LIKE '____';

In [ ]:
%%sql
-- IN: filter for a specific list of values
SELECT * FROM Employees
WHERE last_name IN ('Doe', 'Smith', 'Davis');

In [ ]:
%%sql
-- BETWEEN: inclusive range
SELECT first_name, department_id FROM Employees
WHERE department_id BETWEEN 10 AND 40;

### 🧪 Class Practice 5
1. Employees whose email contains `'example'`.
2. First names with exactly 5 characters.
3. Employees in departments 20 or 50 using `IN`.
4. Salaries between 60k and 80k using `BETWEEN`.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 6 · Aggregations — COUNT · SUM · AVG · GROUP BY · HAVING

Aggregate functions collapse many rows into one number. `GROUP BY` creates one row **per group**. `HAVING` filters **after** aggregation (WHERE filters before).


In [ ]:
%%sql
-- Count of rows in the table
SELECT COUNT(*) FROM Employees;

In [ ]:
%%sql
-- Total of all salaries
SELECT SUM(salary) AS totalsum FROM Employees;

In [ ]:
%%sql
-- Average salary
SELECT AVG(salary) AS averagesalary FROM Employees;

In [ ]:
%%sql
-- MIN and MAX
SELECT MIN(salary) AS lowest, MAX(salary) AS highest FROM Employees;

In [ ]:
%%sql
-- GROUP BY: one row per department
SELECT department_id, COUNT(*) AS employee_count, AVG(salary) AS avg_salary
FROM Employees
GROUP BY department_id;

In [ ]:
%%sql
-- HAVING: filter AFTER aggregation (WHERE cannot use aggregates!)
SELECT department_id, SUM(salary) AS total_salary
FROM Employees
GROUP BY department_id
HAVING SUM(salary) > 140000;

### 💬 Interview Corner — WHERE vs HAVING
- `WHERE` filters **rows before** grouping → cannot contain aggregate functions.
- `HAVING` filters **groups after** aggregation → `HAVING SUM(salary) > 80000` ✅, `WHERE SUM(salary) > 80000` ❌ error.

### 🧪 Class Practice 6
1. Count of employees per department.
2. Highest salary in each department.
3. Departments whose average salary exceeds 75k (needs HAVING).


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 7 · Joins

Combine rows from two (or more) tables based on a related column.

| Join | Result |
|---|---|
| **INNER JOIN** | Only **matching** rows from both tables — a new row per match |
| **LEFT JOIN** | Matches **+ all remaining rows from the LEFT** table (nulls where no match) |
| **RIGHT JOIN** | Matches **+ all remaining rows from the RIGHT** table (nulls where no match) |
| **FULL OUTER JOIN** | Matches + remaining rows from **BOTH** tables |
| **SELF JOIN** | A table joined with itself (managers, match fixtures) |
| **CROSS JOIN** | Cartesian product — every row × every row |

### 🧠 The classic dry-run (memorise this pattern!)
```
Table A: 1, 1, null, 2, null, 4      Table B: 1, 1, 2, null, 3, 2
inner : 1 1 1 1 2 2
left  : 1 1 1 1 2 2 4 null null
right : 1 1 1 1 2 2 3 null
full  : 1 1 1 1 2 2 4 3 null null null
```
Key insight: **each match creates a row** (1s multiply: 2 × 2 = 4 rows) and **null never matches null**.


In [ ]:
%%sql
-- INNER JOIN: employee names with their department names
SELECT e.first_name, e.last_name, d.department_name
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id;

In [ ]:
%%sql
-- LEFT JOIN from Departments: show ALL departments even with no employees
-- ('Support' appears with nulls!)
SELECT d.department_name, e.first_name, e.last_name
FROM Departments d
LEFT JOIN Employees e ON d.department_id = e.department_id;

In [ ]:
%%sql
-- RIGHT JOIN: same result, written from the employee side
-- (requires SQLite 3.39+ / any modern engine)
SELECT e.first_name, d.department_name
FROM Employees e
RIGHT JOIN Departments d ON e.department_id = d.department_id;

In [ ]:
%%sql
-- Multi-table join: name + department + job title + job history dates, only IT
SELECT e.first_name, e.last_name, d.department_name, j.job_title,
       jh.start_date, jh.end_date
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id
JOIN Job_History jh ON e.employee_id = jh.employee_id
JOIN Jobs j ON jh.job_id = j.job_id
WHERE d.department_name = 'IT';

In [13]:
%%sql
-- SELF JOIN setup: an employee table where managerid points back to empid
DROP TABLE IF EXISTS employee_mgr;
CREATE TABLE employee_mgr (
    empid INT PRIMARY KEY,
    empname VARCHAR(100) NOT NULL,
    managerid INT,
    FOREIGN KEY (managerid) REFERENCES employee_mgr(empid)
);
INSERT INTO employee_mgr VALUES
(1, 'Ravi Kumar', NULL),
(2, 'Sneha Sharma', 1),
(3, 'Arjun Mehta', 1),
(4, 'Divya Reddy', 2),
(5, 'Kiran Rao', 2),
(6, 'Manoj Bhat', 3);

[]

In [14]:
%%sql
-- SELF JOIN: every employee with their manager's name
SELECT e.empname AS employeename, m.empname AS managername
FROM employee_mgr e
JOIN employee_mgr m ON e.managerid = m.empid;

employeename,managername
Sneha Sharma,Ravi Kumar
Arjun Mehta,Ravi Kumar
Divya Reddy,Sneha Sharma
Kiran Rao,Sneha Sharma
Manoj Bhat,Arjun Mehta


In [15]:
%%sql
-- ⭐ IPL fixtures: each team plays every other team EXACTLY ONCE
-- The trick: self join on t1.id < t2.id  (removes reverses and self-matches)
DROP TABLE IF EXISTS ipl_teams;
CREATE TABLE ipl_teams (team_id INT PRIMARY KEY, team_name VARCHAR(100));
INSERT INTO ipl_teams VALUES (1,'CSK'), (2,'RCB'), (3,'MI'), (4,'KKR');

SELECT t1.team_name AS teama, t2.team_name AS teamb
FROM ipl_teams t1
JOIN ipl_teams t2 ON t1.team_id < t2.team_id
ORDER BY t1.team_name, t2.team_name;

teama,teamb
CSK,KKR
CSK,MI
CSK,RCB
MI,KKR
RCB,KKR
RCB,MI


In [16]:
%%sql
-- CROSS JOIN (cartesian product): every product in every size
DROP TABLE IF EXISTS products_cj; DROP TABLE IF EXISTS sizes_cj;
CREATE TABLE products_cj (productid INT, productname VARCHAR(50));
CREATE TABLE sizes_cj (sizeid INT, size VARCHAR(10));
INSERT INTO products_cj VALUES (1,'tshirt'), (2,'jeans');
INSERT INTO sizes_cj VALUES (1,'small'), (2,'medium'), (3,'large');

SELECT p.productname, s.size
FROM products_cj p
CROSS JOIN sizes_cj s;
-- 2 products × 3 sizes = 6 rows

[]

### 🧪 Class Practice 7
1. Show each employee with their department **location**.
2. Departments with NO employees (hint: LEFT JOIN from Departments + `WHERE e.employee_id IS NULL`).
3. Employees who earn more than their manager (self join on `employee_mgr` — add salaries first!).
4. A `colors` × `sizes` cross join with 3 colours and 2 sizes → predict the row count before running.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 8 · The C-T-A-G Query-Writing Framework ⭐

**Before writing ANY query, plan it in four lines.** This is the BrowseJobs method for never freezing in an interview:

| Letter | Question | Example |
|---|---|---|
| **C** — Columns | What columns does the answer need? | `d.department_name, avg(e.salary), count(e.employee_id)` |
| **T** — Tables | Which tables hold them? | `employees, departments` |
| **A** — Aggregations | Any SUM/AVG/COUNT/MIN/MAX? | `avg(salary), count(employee_id)` |
| **G** — Grouping | Group by what? | `group by department_name` |

Then the query writes itself:
```sql
select <C> from <T1> join <T2> on primarykey = foreignkey
where <filters> group by <G>
```


In [ ]:
%%sql
-- Worked example: "average salary AND count of employees by department name"
-- C: d.department_name, avg(e.salary), count(e.employee_id)
-- T: Employees, Departments
-- A: avg(salary), count(employee_id)
-- G: group by department_name

SELECT d.department_name,
       AVG(e.salary) AS averagesalary,
       COUNT(e.employee_id) AS countofemployees
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id
GROUP BY d.department_name;

In [5]:
%%sql
-- "Count of employees by EACH department — including empty ones"
-- C: department_name, count(employee_id) | T: both | A: count | G: department_name
-- Trick: LEFT JOIN from Departments so 'Support' still shows (count = 0)

SELECT d.department_name, COUNT(e.employee_id) AS countofemployees
FROM Departments d
LEFT JOIN Employees e ON d.department_id = e.department_id
GROUP BY d.department_name;

department_name,countofemployees
Development,2
Finance,2
Human Resources,2
IT,2
Marketing,2
Support,0


In [6]:
%%sql
-- "All employees who work in the HR department (department_id = 20)"
-- C: first_name, last_name, department_name | T: both | A: none | G: none

SELECT e.first_name, e.last_name, d.department_name
FROM Employees e
JOIN Departments d ON e.department_id = d.department_id
WHERE d.department_id = 20;

first_name,last_name,department_name
Jane,Smith,Human Resources
Olivia,Taylor,Human Resources


### 🧪 Class Practice 8 — plan C-T-A-G first, then write!
1. Total salary paid per location.
2. Job title + count of employees holding it.
3. Department name + highest salary in it, only for departments where that max exceeds 80k.


In [ ]:
%%sql
-- ✏️ Try it here (write your C-T-A-G plan as comments first!)
SELECT 1;

---
# Chapter 9 · Subqueries & CTEs

### Subquery — a query inside a query
Runs the inner query first, feeds its result to the outer one.

### CTE (Common Table Expression) — `WITH name AS (...)`
An optimised, readable way to structure SQL. The result lives in a **virtual table** that exists ONLY for the execution of the statement — it does not save the data OR the code.

**Advantages:** easy to read and manage, optimised, perfect for filtering on window functions and aggregates.


In [ ]:
%%sql
-- Subquery: employees earning MORE than the average salary
SELECT first_name, last_name, salary
FROM Employees
WHERE salary > (SELECT AVG(salary) FROM Employees);

In [ ]:
%%sql
-- Subquery: employee(s) with the newest hire date
SELECT first_name, last_name, hire_date
FROM Employees
WHERE hire_date = (SELECT MAX(hire_date) FROM Employees);

In [ ]:
%%sql
-- Or with ORDER BY + LIMIT: first person ever hired
SELECT first_name, last_name, hire_date
FROM Employees
ORDER BY hire_date
LIMIT 1;

In [ ]:
%%sql
-- Top 2 salaries
SELECT first_name, last_name, salary
FROM Employees
ORDER BY salary DESC
LIMIT 2;

In [ ]:
%%sql
-- CTE: salary > 70k AND name starts with a vowel-ish filter, highest first
WITH greaterthan70 AS (
    SELECT employee_id, first_name, last_name, salary
    FROM Employees
    WHERE salary > 70000
)
SELECT * FROM greaterthan70
WHERE first_name LIKE 'J%' OR last_name LIKE 'J%'
ORDER BY salary DESC;

In [ ]:
%%sql
-- CTE + aggregation: departments whose TOTAL salary exceeds 140k
-- (this is where WHERE fails and CTE/HAVING shines)
WITH cte AS (
    SELECT department_id, SUM(salary) AS total_salary
    FROM Employees
    GROUP BY department_id
)
SELECT * FROM cte
WHERE total_salary > 140000;

In [ ]:
%%sql
-- CTE: employees hired after 2020-01-01 who earn above 75k — layered filtering
WITH cte AS (
    SELECT employee_id, first_name, last_name, hire_date, salary
    FROM Employees
    WHERE hire_date > '2020-01-01'
)
SELECT * FROM cte
WHERE salary > 75000;

### 🧪 Class Practice 9
1. Employees earning below the company average (subquery).
2. CTE of Finance employees → from it, pick the highest paid.
3. Departments where the average salary is above the **company** average (CTE + subquery combo).


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 10 · Window Functions

Aggregate functions that work **at row level** — every row keeps its identity but gains an aggregate/ranked column. Syntax:

```sql
function() OVER (PARTITION BY col ORDER BY col)
```

### RANK vs DENSE_RANK vs ROW_NUMBER (top interview question!)
| Function | On ties... |
|---|---|
| `RANK()` | Same rank, then **SKIPS** (1,1,3) |
| `DENSE_RANK()` | Same rank, **NO skip** (1,1,2) |
| `ROW_NUMBER()` | Unique number regardless (1,2,3) |

```
salary  rank  denserank  rownumber
10000    1        1          1
 8000    2        2          2
 8000    2        2          3
 5000    4        3          4
 5000    4        3          5
 2000    6        4          6
```

### LAG and LEAD
- `LAG(col)` — value from the **previous** row.
- `LEAD(col)` — value from the **next** row.


In [ ]:
%%sql
-- See all three ranking functions side by side
SELECT first_name, salary,
       RANK()       OVER (ORDER BY salary DESC) AS rnk,
       DENSE_RANK() OVER (ORDER BY salary DESC) AS denserank,
       ROW_NUMBER() OVER (ORDER BY salary DESC) AS rownumber
FROM Employees;

In [ ]:
%%sql
-- ⭐ SECOND HIGHEST SALARY — the most asked SQL interview question
-- Method 1: CTE + DENSE_RANK
WITH cte AS (
    SELECT employee_id, first_name, last_name, salary,
           DENSE_RANK() OVER (ORDER BY salary DESC) AS rankings
    FROM Employees
)
SELECT * FROM cte
WHERE rankings = 2;

In [ ]:
%%sql
-- Method 2: ORDER BY + LIMIT/OFFSET
SELECT first_name, last_name, salary
FROM Employees
ORDER BY salary DESC
LIMIT 1 OFFSET 1;

In [ ]:
%%sql
-- ⭐ Second highest salary BY DEPARTMENT → add PARTITION BY
WITH cte AS (
    SELECT department_id, employee_id, first_name, last_name, salary,
           DENSE_RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS rankings
    FROM Employees
)
SELECT * FROM cte
WHERE rankings = 2;

In [ ]:
%%sql
-- LEAD: each employee's salary next to the NEXT hire's salary
SELECT employee_id, first_name, hire_date, salary,
       LEAD(salary) OVER (ORDER BY hire_date) AS nextemployeesalary
FROM Employees;

In [ ]:
%%sql
-- LAG: compare with the PREVIOUS employee in the same department
SELECT employee_id, first_name, department_id, salary,
       LAG(salary) OVER (PARTITION BY department_id ORDER BY salary) AS previoussalary
FROM Employees;

In [ ]:
%%sql
-- LAG arithmetic: salary GROWTH vs previous hire in the department
SELECT employee_id, first_name, department_id, salary,
       salary - LAG(salary) OVER (PARTITION BY department_id ORDER BY hire_date) AS salary_growth
FROM Employees;

In [ ]:
%%sql
-- Both together: previous AND next salary by hire date
SELECT employee_id, first_name, salary,
       LAG(salary)  OVER (ORDER BY hire_date) AS prevsalary,
       LEAD(salary) OVER (ORDER BY hire_date) AS next_salary
FROM Employees;

In [ ]:
%%sql
-- Real scenario: salary increments per employee over time
DROP TABLE IF EXISTS emp_increments;
CREATE TABLE emp_increments (
    employee_id INT, first_name VARCHAR(50), salary DECIMAL(10,2), hire_date DATE
);
INSERT INTO emp_increments VALUES
(101, 'Alice', 50000, '2020-01-15'),
(101, 'Alice', 55000, '2021-01-15'),
(101, 'Alice', 60000, '2022-01-15'),
(102, 'Bob', 45000, '2019-03-10'),
(102, 'Bob', 47000, '2020-03-10'),
(102, 'Bob', 52000, '2021-03-10'),
(103, 'Charlie', 60000, '2021-06-20'),
(103, 'Charlie', 63000, '2022-06-20'),
(104, 'David', 70000, '2020-09-01');

WITH cte AS (
    SELECT employee_id, first_name, salary,
           LEAD(salary) OVER (PARTITION BY employee_id ORDER BY hire_date) AS incrementedsalary
    FROM emp_increments
)
SELECT employee_id, first_name, salary, incrementedsalary,
       ROUND(((incrementedsalary - salary) / salary) * 100, 2) AS percentageincrease
FROM cte;

In [ ]:
%%sql
-- Real scenario: total logged-in time + session count per employee
DROP TABLE IF EXISTS logtable;
CREATE TABLE logtable (
    logid INTEGER PRIMARY KEY AUTOINCREMENT,
    employeeid INT, logintime TIMESTAMP, logouttime TIMESTAMP
);
INSERT INTO logtable (employeeid, logintime, logouttime) VALUES
(101, '2025-09-01 09:00:00', '2025-09-01 11:00:00'),
(101, '2025-09-02 10:00:00', '2025-09-02 13:30:00'),
(101, '2025-09-03 09:15:00', '2025-09-03 12:15:00'),
(102, '2025-09-01 08:45:00', '2025-09-01 09:45:00'),
(102, '2025-09-02 09:00:00', '2025-09-02 10:30:00'),
(102, '2025-09-03 10:00:00', '2025-09-03 11:00:00'),
(103, '2025-09-01 09:30:00', '2025-09-01 14:30:00'),
(103, '2025-09-02 09:00:00', '2025-09-02 12:00:00');

SELECT employeeid,
       COUNT(*) AS sessioncount,
       SUM(strftime('%s', logouttime) - strftime('%s', logintime)) AS totaltimeseconds
FROM logtable
GROUP BY employeeid
ORDER BY totaltimeseconds DESC;

### 🧪 Class Practice 10
1. THIRD highest salary company-wide.
2. Rank employees by salary WITHIN each department using all three ranking functions — study the differences.
3. Days between hires: `LAG(hire_date)` + date difference (SQLite: `julianday(hire_date) - julianday(prev)`).


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 11 · CASE Statements & Stored Procedures

### CASE — SQL's if/elif/else
```sql
CASE WHEN condition THEN value
     WHEN condition THEN value
     ELSE value
END AS newcolumn
```


In [ ]:
%%sql
-- Bonus by role: DEV01 → 10%, HR01 → 15%, everyone else 0
SELECT first_name, job_id, salary,
       CASE WHEN job_id = 'DEV01' THEN salary * 0.1
            WHEN job_id = 'HR01'  THEN salary * 0.15
            ELSE 0
       END AS bonus
FROM Employees;

# if job_id='DEV01':
#   salary*0.1
# elif job_id='HR01':
#   salary*0.15
# else:
#   0

first_name,job_id,salary,bonus
John,DEV01,75000,7500.0
Jane,HR01,68000,10200.0
Robert,FIN01,85000,0
Emily,MKT01,62000,0
Michael,DEV02,78000,0
Olivia,HR02,71000,0
William,FIN02,90000,0
Sophia,MKT02,65000,0
James,IT01,87000,0
Isabella,IT02,82000,0


In [ ]:
%%sql
-- Categorise employees by salary band
SELECT first_name, salary,
       CASE WHEN salary >= 85000 THEN 'high'
            WHEN salary BETWEEN 63000 AND 84999 THEN 'medium'
            ELSE 'low'
       END AS salarycategory
FROM Employees;

first_name,salary,salarycategory
John,75000,medium
Jane,68000,medium
Robert,85000,high
Emily,62000,low
Michael,78000,medium
Olivia,71000,medium
William,90000,high
Sophia,65000,medium
James,87000,high
Isabella,82000,medium


### Stored Procedures — reference (SQL Server / MySQL syntax; not supported in SQLite)

A **stored procedure** = a group of SQL statements saved as a single named unit.
**Why:** faster execution (pre-compiled) · prevents SQL-injection attacks · reusable without rewriting.

**SQL Server style (@ parameters):**
```sql
CREATE PROCEDURE updateemployeesalary
    @employee_id INT,
    @newsalary DECIMAL(10,2)
AS BEGIN
    UPDATE employees
    SET salary = @newsalary
    WHERE employee_id = @employee_id;
END;

EXEC updateemployeesalary @employee_id = 105, @newsalary = 70000.00;
EXEC updateemployeesalary @employee_id = 106, @newsalary = 70000.00;
```

**MySQL style (IN parameters):**
```sql
CREATE PROCEDURE addemployee(
    IN empid INT, IN empname VARCHAR(255),
    IN dept VARCHAR(255), IN salary INT)
BEGIN
    INSERT INTO employees (id, name, department, salary)
    VALUES (empid, empname, dept, salary);
END;

CALL addemployee(10, 'john', 'IT', 50000);
```

**Bulk update with CASE inside a procedure:**
```sql
UPDATE employees SET salary =
CASE WHEN employeeid BETWEEN 1  AND 10 THEN 50000
     WHEN employeeid BETWEEN 11 AND 20 THEN 60000
     WHEN employeeid BETWEEN 21 AND 30 THEN 70000
END
WHERE employeeid IN (...);
```

**Multiple ids as one parameter (SQL Server):** pass `'101,103,105'` and filter with
`WHERE employee_id IN (SELECT value FROM string_split(@employee_id, ','))`.


### 🧪 Class Practice 11
1. Add a `seniority` column with CASE: hired before 2019 → 'veteran', 2019-2021 → 'mid', after → 'new'.
2. Write (on paper) a MySQL procedure `showemployeesbydepartment(IN deptname)` that selects employees for a given department.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 12 · DELETE vs TRUNCATE vs DROP · UNION · Duplicates

### The destruction hierarchy (interview staple)
| Command | Removes | Rollback? |
|---|---|---|
| `DELETE FROM t WHERE ...` | **Specific rows** based on a condition | ✅ (DML, logged) |
| `TRUNCATE TABLE t` | **All rows** — table structure stays | ⚠️ Usually not |
| `DROP TABLE t` | The **entire table** (or database!) | ❌ Gone |

Also: `ALTER TABLE employee DROP COLUMN salary;` removes a **column**.

### UNION vs UNION ALL
- `UNION` — combines result sets and **removes duplicates**.
- `UNION ALL` — combines and **keeps duplicates** (faster — no dedup step).
- Column counts must match; pad with `NULL AS colname` if needed.


In [ ]:
%%sql
-- UNION demo: cities from two tables
DROP TABLE IF EXISTS emp_cities; DROP TABLE IF EXISTS dept_cities;
CREATE TABLE emp_cities (city VARCHAR(50));
CREATE TABLE dept_cities (city VARCHAR(50));
INSERT INTO emp_cities VALUES ('bangalore'),('chennai'),('delhi'),('mysore'),('bangalore'),('delhi');
INSERT INTO dept_cities VALUES ('bangalore'),('chennai'),('delhi'),('mysore'),('trivandrum');

-- UNION: duplicates removed → 5 unique cities
SELECT city FROM emp_cities
UNION all
SELECT city FROM dept_cities;

city
bangalore
chennai
delhi
mysore
bangalore
delhi
bangalore
chennai
delhi
mysore


In [ ]:
%%sql
-- UNION ALL: duplicates kept → all 11 rows
SELECT city FROM emp_cities
UNION ALL
SELECT city FROM dept_cities;

### Handling duplicates — the 3 techniques
1. **Find** them: `GROUP BY ... HAVING COUNT(*) > 1`
2. **Hide** them: `SELECT DISTINCT`
3. **Delete** them: CTE + `ROW_NUMBER()` partitioned by the duplicate columns → delete `rownumber > 1`


In [ ]:
%%sql
-- Setup: a table with duplicate people
DROP TABLE IF EXISTS emp_dupes;
CREATE TABLE emp_dupes (employee_id INT, first_name VARCHAR(50), last_name VARCHAR(50), email VARCHAR(100));
INSERT INTO emp_dupes VALUES
(1,'krish','bhargav','krish@gmail.com'),
(2,'krish','bhargav','krish@gmail.com'),
(3,'krish','bhargav','krish@gmail.com'),
(4,'adveer','bhargav','adveer@gmail.com'),
(5,'adveer','bhargav','adveer@gmail.com'),
(6,'araia','bhargav','araia@gmail.com');

-- 1) FIND duplicates
SELECT email, COUNT(*) AS occurrences
FROM emp_dupes
GROUP BY email
HAVING COUNT(*) > 1;

email,occurrences
adveer@gmail.com,2
krish@gmail.com,3


In [ ]:
%%sql
-- 2) HIDE duplicates

SELECT DISTINCT email FROM emp_dupes;

email
krish@gmail.com
adveer@gmail.com
araia@gmail.com


In [ ]:
%%sql
-- 3) DELETE duplicates, keeping row 1 of each group
-- create read update delete (CRUD)
DELETE FROM emp_dupes
WHERE employee_id IN (
    SELECT employee_id FROM (
        SELECT employee_id,
               ROW_NUMBER() OVER (PARTITION BY first_name, last_name, email
                                  ORDER BY employee_id) AS rownumber
        FROM emp_dupes
    )
    WHERE rownumber > 1
);

SELECT * FROM emp_dupes;   -- clean!

employee_id,first_name,last_name,email
1,krish,bhargav,krish@gmail.com
4,adveer,bhargav,adveer@gmail.com
6,araia,bhargav,araia@gmail.com


### 🧪 Class Practice 12
1. Re-insert duplicates and delete them keeping the LAST id instead of the first (change the ORDER BY).
2. UNION the department locations with the cities table — how many rows with UNION vs UNION ALL?


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 13 · Query Optimisation

### The 7 optimisation techniques
1. **Indexing** — retrieve data quickly from frequently-queried columns (⚠️ over-indexing hurts writes).
2. **Select named columns**, not `SELECT *`.
3. **Use proper joins** — prefer INNER JOIN where possible; use WHERE to limit data early.
4. **Query execution plan** — `EXPLAIN` / `ANALYZE` to see how the query is processed.
5. **Partitioning / sharding** — divide large tables (range or hash partitioning; hash example: `122 % 10 = partition 2`).
6. **CTEs** — structure without storing data.
7. **Materialized views** — for large tables used in repeated computation.

### Indexing deep-dive
- **B-Tree index** — self-balancing tree; enables **index seek** (jump straight to the row's location instead of a full table scan).
- **Hash index** — uses a hash function for exact-match lookups.
- **Clustered index** — only ONE per table (usually the primary key); **physically re-orders** the table rows on disk; the B-tree **leaf nodes contain the data**.
- **Non-clustered index** — many allowed; a **separate structure** whose leaf nodes contain **pointers (row locators)** to the data; physical order unchanged.

### Views vs Materialized Views
| | View | Materialized View |
|---|---|---|
| Stores | Only the **query** | The query **AND its results** on disk |
| Freshness | Always live | Needs **refresh** |
| Speed | Runs query each time | Instant reads |

**Refresh strategies:** manual (`REFRESH MATERIALIZED VIEW ...`) · scheduled (cron) · fast/incremental (only changed data, via logs).

### Cron expressions — `min hour day-of-month month day-of-week`
| Expression | Schedule |
|---|---|
| `* * * * *` | Every minute |
| `0 * * * *` | Every hour |
| `0 0 * * *` | Every day at 12:00 AM |
| `0 0 * * FRI` | 12:00 AM, only Friday |
| `0 0 1 * *` | 12:00 AM, day 1 of the month |


In [ ]:
%%sql
-- Create an index and inspect the query plan
CREATE INDEX IF NOT EXISTS idx_salary ON Employees(salary);

EXPLAIN QUERY PLAN
SELECT * FROM Employees WHERE salary > 75000;
-- Look for "USING INDEX idx_salary" instead of "SCAN"!

[]

In [ ]:
%%sql
-- Multi-column (composite) index
CREATE INDEX IF NOT EXISTS idx_firstname_lastname ON Employees(first_name, last_name);

EXPLAIN QUERY PLAN
SELECT * FROM Employees WHERE first_name = 'John' AND last_name = 'Doe';

id,parent,notused,detail
3,0,0,SEARCH Employees USING INDEX idx_firstname_lastname (first_name=? AND last_name=?)


In [ ]:
%%sql
-- Views: a virtual table that stores only the query
DROP VIEW IF EXISTS highearners;
CREATE VIEW highearners AS
    SELECT employee_id, first_name, last_name, salary
    FROM Employees
    WHERE salary > 75000;

SELECT * FROM highearners;

employee_id,first_name,last_name,salary
105,Michael,Wilson,78000
110,Isabella,Jackson,82000
103,Robert,Brown,85000
109,James,Thomas,87000
107,William,Moore,90000


### Partitioning — reference (MySQL syntax)
```sql
CREATE TABLE orders (
  orderid INT, customerid INT, orderdate DATE)
PARTITION BY RANGE (YEAR(orderdate)) (
  PARTITION p2022 VALUES LESS THAN (2023),
  PARTITION p2023 VALUES LESS THAN (2024),
  PARTITION pmax  VALUES LESS THAN MAXVALUE);

SELECT * FROM orders WHERE YEAR(orderdate) = 2023;  -- scans only p2023!
```

### ⭐ Order of Execution — "**F**rom **J**apan **O**nce **W**ise **G**irl **H**as **S**elected **D**istinct **O**range **L**addus"
```
FROM → JOIN → ON → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT
```
The fruit-basket analogy: take the baskets (FROM), combine them (JOIN/ON), remove fruits you don't need (WHERE), group them (GROUP BY), filter groups (HAVING), pick the ones to show (SELECT), remove repeats (DISTINCT), arrange (ORDER BY), take only the top few (LIMIT).

**This is why:** you can't use a SELECT alias in WHERE (WHERE runs first) but you CAN in ORDER BY (runs after).


In [ ]:
select * from orders partition (p2022);

In [ ]:
select * from employees
where salary > 7000
order by salary desc
limit 1;

### 🧪 Class Practice 13
1. Run `EXPLAIN QUERY PLAN` on a first_name search before and after creating its index.
2. Create a view `itemployees` (IT department only) and query it with an extra salary filter.
3. Recite the order of execution — then explain why `HAVING` exists at all.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# Chapter 14 · Data Modelling — OLTP vs OLAP · Star vs Snowflake · Normalisation

### OLTP vs OLAP
| | **OLTP** — Online Transaction Processing | **OLAP** — Online Analytical Processing |
|---|---|---|
| Used in | Databases (MySQL, Postgres, SQL Server) | Data warehouses (Redshift, Snowflake) |
| Data | Day-to-day transactions | Transformed, aggregated data |
| Normalisation | **Highly normalised** (reduce redundancy) | **Less normalised** (read faster) |
| Operations | Frequent INSERT/UPDATE/DELETE | Fast analytical reads |
| Schema style | Snowflake-ish | **Star** |

### Star schema ⭐
- A **central fact table** (quantitative: sales, revenue, quantity) surrounded by **dimension tables** (descriptive: customer, product, category) — the diagram looks like a star.
- Less normalised → **faster retrieval** → the data-warehouse favourite.

### Snowflake schema ❄️
- A more **normalised** version of star: dimensions have **sub-dimensions** (product → category → subcategory; customer → address → country).
- Better organised, less redundant, easier integrity — but more complex to design and **slower** to query (more joins).

### Normalisation — 1NF → 2NF → 3NF
Breaking one large table into small related tables to remove redundancy.

| Form | Rule | Fixes |
|---|---|---|
| **1NF** | Every cell **atomic** — one value per cell, no repeating groups | `Courses = "Math, Science"` → one row per course |
| **2NF** | 1NF + every non-key column depends on the **whole** primary key | Removes **partial dependencies** |
| **3NF** | 2NF + no column depends on another **non-key** column | Removes **transitive dependencies** (A→B and B→C means A→C ❌) |

Worked example: `Student(id, name, course, instructor)` →
1NF: one course per row → 2NF: split into Student, Enrollments, Course → 3NF: instructor moves out of Course into its own Instructor table (course → instructor was transitive).

### The 3 data-modelling levels
1. **Conceptual** — high-level entities and relationships, no implementation detail (early design).
2. **Logical** — detailed structure of relationships and attributes.
3. **Physical** — actual storage: table names, columns, data types, indexing, performance tuning.

### How a database project team works (real-world context)
Product Manager (CEO of the product) → Scrum Master → Business Analyst gathers requirements → **Solutions Architect** designs end-to-end (tech stack, cloud, database) → **Data Architect** designs the schema (tables, star/snowflake, relations) → Data Architect + **DBA** implement it → staging → production → backup → extract & report.


In [ ]:
from ast import MatchSingleton
#id, name, courses
1, a, math, science
2, b, science, history
3, c, math, history

id name course
1, a, math
1, a, science
2, b, science
2, b, history
3, c, math
3, c, history

In [ ]:
from os import name
from IPython.core.prefilter import PythonOpsChecker
from IPython.core.prefilter import PythonOpsChecker
#2nf : it should comply with 1nf

studentid | studentname| phone  | courses | trainer
1 , a, 987987997, python, sql , amit
2, b, 4654654743, Python, neha
3, c, 9798997987, aws, python, amit

1nf:
studentid | studentname| phone  | course | trainer
1, a, 987987997, python, amit
1, a, 987987997, sql, neha
2, b, 4654654743, Python, amit
3, c, 9798997987, aws, amit
3, c, 9798997987, python,amit

2nf:
studentid, studentname, phone, courseid
1, a, 987987997
2, b, 4654654743
3, c, 9798997987

course
coursid , coursename,trainer
1 , python, amit
2, sql, neha
3, aws, amit


3nf:
studentid, studentname, phone
1, a, 987987997
2, b, 4654654743
3, c, 9798997987

course
coursid , coursename,trainerid
1 , python, 1
2, sql, 2

trainer:
trainerid , name
1, ajay
2, neha

A>b>C

In [ ]:
from inspect import AGEN_CLOSED
id , name , age
1, a, 16
2, b, 18
3, c, 21

id, name, age
4, d, 21
5, e, 22
3, c, 21
2, b, 18

union: merge 2 table but removes duplicates

id, name, age
1, a, 16
2, b, 18
3, c, 21
4, d, 21
5, e, 22


unionall : merge 2 table but keeps duplicates
id , name , age
1, a, 16
2, b, 18
3, c, 21
4, d, 21
5, e, 22
3, c, 21
2, b, 18



In [ ]:
from inspect import ClosureVars
from re import L
Cartesian or cross join

size
id, size
1, S
2, M
3, L

Colors :
id, colour
1, black
2, white
3, blue


select s.size, c.colour from size s cross join colors c



s black
s white
s blue
m black
m white
m blue
l black
l white
l blue





### 💬 Interview Corner — the SQL topic checklist
Complex joins · CTEs · indexing · normalisation · OLTP vs OLAP · star vs snowflake · fact vs dimension · optimisation techniques · window functions · WHERE vs HAVING · views vs materialized views · order of execution · CASE statements · stored procedures · DELETE/TRUNCATE/DROP · UNION vs UNION ALL · self join · cartesian join · duplicates handling · second-highest salary · LEAD/LAG · subqueries.

If you can whiteboard **second-highest salary by department** and recite the **order of execution**, you clear most L1 SQL rounds.

### 🧪 Final Class Practice
1. Draw a star schema for BrowseJobs: fact = enrollments; dimensions = student, course, trainer, payment.
2. Normalise `Student(id, name, courses="math,science", instructors="Dr A, Dr B")` to 3NF on the whiteboard.
3. Mock interview pairs: one asks the checklist above, the other answers — then swap.


In [ ]:
%%sql
-- ✏️ Try it here
SELECT 1;

---
# 🎓 Course Recap — One-Slide Summary

| Chapter | Key takeaway |
|---|---|
| 01 Databases | SQL = tables/fixed schema/vertical · NoSQL = flexible/horizontal · CRUD |
| 02 Keys | PK unique+not null · FK duplicates+nulls OK · candidate/surrogate/natural/composite |
| 03 DDL | CREATE TABLE + constraints · INSERT INTO ... VALUES |
| 04 SELECT | WHERE · AND/OR/NOT · ORDER BY (ASC default) · LIMIT · AS |
| 05 Filters | LIKE `%` `_` · IN (list) · BETWEEN (inclusive) |
| 06 Aggregates | COUNT/SUM/AVG/MIN/MAX · GROUP BY · HAVING filters AFTER aggregation |
| 07 Joins | inner=matches · left/right=+remainder · each match = new row · null≠null |
| 08 C-T-A-G | Columns → Tables → Aggregations → Grouping, THEN write |
| 09 Subqueries/CTE | `WITH cte AS (...)` — virtual, lasts one execution |
| 10 Windows | RANK skips, DENSE_RANK doesn't, ROW_NUMBER unique · LAG prev · LEAD next |
| 11 CASE/Procs | CASE WHEN...THEN...ELSE...END · procs = fast, reusable, injection-safe |
| 12 Cleanup | DELETE row · TRUNCATE all rows · DROP table · UNION dedupes, UNION ALL doesn't |
| 13 Optimisation | index seek > table scan · clustered=1/physical · view=query, matview=query+data |
| 14 Modelling | OLTP normalised, OLAP star · 1NF atomic, 2NF whole-key, 3NF no transitive |

**Order of execution:** FROM → JOIN → ON → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT

*Built for BrowseJobs Technologies · SQL Training Track*
